# Dependency trees of EstNLTK texts

`DependencyTreeVisualiser` draws the dependency trees of a `Text` as SVG diagrams with
rectangular arcs. Under the words it can show annotation rows from any layer (e.g.
part of speech and form from `morph_analysis`) and boxes over the spans of a layer
(e.g. named entities). A highlight marks a fragment of a sentence, and `window`
limits the diagram to that fragment.

In [1]:
from estnltk import Text

from estnltk_syntax_vis import DependencyTreeVisualiser, Style

## A text with a syntax layer

The visualiser needs a syntax layer with the attributes `id`, `head` (0 for the root)
and `deprel`. Here it is EstNLTK's `maltparser_syntax`, which runs out of the box
(it needs Java). Morphology and named entities come from EstNLTK's standard taggers.

`StanzaSyntaxTagger` from `estnltk_neural` parses more accurately; its layer
`stanza_syntax` is the visualiser's default `syntax_layer`.

In [2]:
text = Text(
    "Eesti Vabariigi president Alar Karis kohtus eile Tallinnas Soome peaministriga. "
    "Kohtumisel arutati Tallinna ja Helsingi vahelist tunnelit."
)
text.tag_layer(["maltparser_syntax", "ner"])

Text(text='Eesti Vabariigi president Alar Karis kohtus eile Tallinnas Soome peaministriga. Kohtumisel arutati Tallinna ja Helsingi vahelist tunnelit.')

## Whole sentences

Configure the visualiser once and apply it to any number of texts. Calling it displays
one diagram per sentence.

In [3]:
vis = DependencyTreeVisualiser(syntax_layer="maltparser_syntax")
vis(text)

root 
 nmod 
 nmod 
 nsubj 
 appos 
 flat 
 advmod 
 obl 
 nmod 
 obl 
 punct 
 Eesti 
 Vabariigi 
 president 
 Alar 
 Karis 
 kohtus 
 eile 
 Tallinnas 
 Soome 
 peaministriga 
 . 
 
 
 
 
 
 
 
 
 
 
 
 
 
 root 
 obl 
 obl 
 cc 
 conj 
 amod 
 obj 
 punct 
 Kohtumisel 
 arutati 
 Tallinna 
 ja 
 Helsingi 
 vahelist 
 tunnelit 
 .

## Annotation rows and boxes

`rows` lists what to show under the words. A row is `(layer, attribute)`, matched to
the words by text position, so any layer over the same words works. `overlay_layer`
draws a box over the words of each span, labelled with the attribute `overlay_label`.

In [4]:
vis = DependencyTreeVisualiser(
    syntax_layer="maltparser_syntax",
    rows=[("morph_analysis", "partofspeech"), ("morph_analysis", "form")],
    overlay_layer="ner",
    overlay_label="nertag",
)
vis(text)

root 
 nmod 
 nmod 
 nsubj 
 appos 
 flat 
 advmod 
 obl 
 nmod 
 obl 
 punct 
 Eesti 
 H 
 sg g 
 Vabariigi 
 S 
 sg g 
 president 
 S 
 sg n 
 Alar 
 H 
 sg n 
 Karis 
 H 
 sg n 
 kohtus 
 V 
 s 
 eile 
 D 
 
 Tallinnas 
 H 
 sg in 
 Soome 
 H 
 sg g 
 peaministriga 
 S 
 sg kom 
 . 
 Z 
 
 LOC 
 PER 
 LOC 
 LOC 
 
 partofspeech 
 form 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 root 
 obl 
 obl 
 cc 
 conj 
 amod 
 obj 
 punct 
 Kohtumisel 
 S 
 sg ad 
 arutati 
 V 
 ti 
 Tallinna 
 H 
 sg g 
 ja 
 J 
 
 Helsingi 
 H 
 sg g 
 vahelist 
 A 
 sg p 
 tunnelit 
 S 
 sg p 
 . 
 Z 
 
 LOC 
 LOC 
 
 partofspeech 
 form

## A relation layer as the overlay

`overlay_layer` can also be an EstNLTK `RelationLayer`. Each named span of each relation
becomes a box labelled `name (index)`, where the index is the relation's position in the
layer, so spans of the same relation can be matched up. `overlay_label` names a relation
attribute, shown as a second line in the box.

Spans often overlap: below, "Mari" and "Sipsikut" are each the `entity` of two
coreference relations (example from EstNLTK's RelationLayer tutorial). Overlapping boxes
get their own label lines, nested boxes enclose each other, and the semi-transparent
fills make overlapping regions darker. The same applies to overlapping spans of a span
layer.

In [5]:
from estnltk_core import RelationLayer

coref_text = Text(
    'Mari kirjeldas õhinal, kuidas ta väiksena "Sipsikut" luges: "Ma ei suutnud '
    'seda raamatut kohe kuidagi käest ära panna! Nii põnev oli see!"'
).tag_layer(["maltparser_syntax"])
coref = RelationLayer(
    "coreference", span_names=["mention", "entity"], text_object=coref_text
)
coref.add_annotation(mention=(30, 32), entity=(0, 4))
coref.add_annotation(mention=(61, 63), entity=(0, 4))
coref.add_annotation(mention=(75, 88), entity=(42, 52))
coref.add_annotation(mention=(133, 136), entity=(42, 52))
coref_text.add_layer(coref)

DependencyTreeVisualiser(syntax_layer="maltparser_syntax", overlay_layer="coreference")(
    coref_text
)

root 
 nsubj 
 advmod 
 punct 
 mark 
 nsubj 
 xcomp 
 punct 
 obj 
 punct 
 ccomp 
 punct 
 punct 
 nsubj 
 aux 
 conj 
 det 
 obj 
 advmod 
 advmod 
 obl 
 compound:prt 
 xcomp 
 punct 
 Mari 
 kirjeldas 
 õhinal 
 , 
 kuidas 
 ta 
 väiksena 
 " 
 Sipsikut 
 " 
 luges 
 : 
 " 
 Ma 
 ei 
 suutnud 
 seda 
 raamatut 
 kohe 
 kuidagi 
 käest 
 ära 
 panna 
 ! 
 entity (1) 
 entity (3) 
 mention (0) 
 entity (0) 
 mention (1) 
 mention (2) 
 entity (2) 
 
 
 
 
 
 
 
 
 
 
 
 
 root 
 advmod 
 cop 
 nsubj:cop 
 punct 
 punct 
 Nii 
 põnev 
 oli 
 see 
 ! 
 " 
 mention (3)

## Highlights and fragments

`highlight` marks words: their arcs are drawn in colour and the arcs that attach them
to the rest of the sentence are dashed. It can be

- a span, e.g. a named entity `text.ner[1]` (the sentence is found from its position),
- several spans, e.g. `text.words[3:5]` or a list of words,
- word indices within a sentence, together with `sentence=`.

Only the sentence with the highlight is drawn.

In [6]:
vis(text, highlight=text.ner[1])

root 
 nmod 
 nmod 
 nsubj 
 advmod 
 obl 
 nmod 
 obl 
 punct 
 appos 
 flat 
 Eesti 
 H 
 sg g 
 Vabariigi 
 S 
 sg g 
 president 
 S 
 sg n 
 Alar 
 H 
 sg n 
 Karis 
 H 
 sg n 
 kohtus 
 V 
 s 
 eile 
 D 
 
 Tallinnas 
 H 
 sg in 
 Soome 
 H 
 sg g 
 peaministriga 
 S 
 sg kom 
 . 
 Z 
 
 LOC 
 PER 
 LOC 
 LOC 
 
 partofspeech 
 form

With `window=0` only the highlighted words are shown, together with the heads they
attach to; hidden words become "…". A larger window keeps that many words on each
side of every highlighted word.

In [7]:
vis(text, highlight=text.ner[1], window=0)

appos 
 flat 
 … 
 president 
 S 
 sg n 
 Alar 
 H 
 sg n 
 Karis 
 H 
 sg n 
 … 
 PER 
 
 partofspeech 
 form

In [8]:
vis(text, highlight=[text.words[2], text.words[7]], window=0)

root 
 nsubj 
 obl 
 … 
 president 
 S 
 sg n 
 … 
 kohtus 
 V 
 s 
 … 
 Tallinnas 
 H 
 sg in 
 … 
 LOC 
 
 partofspeech 
 form

In [9]:
vis(text, sentence=1, highlight=[2, 3, 4], window=1)

root 
 obl 
 cc 
 conj 
 … 
 arutati 
 V 
 ti 
 Tallinna 
 H 
 sg g 
 ja 
 J 
 
 Helsingi 
 H 
 sg g 
 vahelist 
 A 
 sg p 
 … 
 LOC 
 LOC 
 
 partofspeech 
 form

## Rows computed by a function

A row can also be `(label, function)` or just a function. The function gets the word's
span of the syntax layer and returns the value to show. Here the part of speech and
case are combined from `morph_analysis`. For ambiguous layers the first annotation is
shown; `ambiguous="all"` shows all distinct values joined with `|`.

In [10]:
def pos_case(word):
    morph = text.morph_analysis.get(word.base_span)
    pos, form = morph.annotations[0]["partofspeech"], morph.annotations[0]["form"]
    return f"{pos}:{form.split()[-1]}" if form else pos


vis_case = DependencyTreeVisualiser(
    syntax_layer="maltparser_syntax",
    rows=[
        ("lemma", lambda word: text.morph_analysis.get(word.base_span).lemma[0]),
        ("POS:case", pos_case),
    ],
)
vis_case(text, highlight=text.words[2:5], window=0)

root 
 nsubj 
 appos 
 flat 
 … 
 president 
 president 
 S:n 
 Alar 
 Alar 
 H:n 
 Karis 
 Karis 
 H:n 
 kohtus 
 kohtuma 
 V:s 
 … 
 
 lemma 
 POS:case

## Styling

Colours, font and sizes are fields of `Style`. Diagrams have their own white
background, so they also work with dark notebook themes.

In [11]:
compact = Style(
    word_font_size=12,
    level_height=16,
    leg_gap=6,
    highlight_fill="#d3f9d8",
    highlight_arc="#2b8a3e",
)
DependencyTreeVisualiser(
    syntax_layer="maltparser_syntax",
    rows=[("morph_analysis", "partofspeech")],
    style=compact,
)(text, sentence=1)

root 
 obl 
 obl 
 cc 
 conj 
 amod 
 obj 
 punct 
 Kohtumisel 
 S 
 arutati 
 V 
 Tallinna 
 H 
 ja 
 J 
 Helsingi 
 H 
 vahelist 
 A 
 tunnelit 
 S 
 . 
 Z 
 
 partofspeech

## SVG and HTML output

`svg` returns the SVG string of one sentence, `svgs` a list for several and
`html_output` the HTML that calling the visualiser displays. SVG files open in
browsers and vector graphics editors.

In [12]:
import tempfile
from pathlib import Path

svg = vis.svg(text, highlight=text.ner[1], window=0)
path = Path(tempfile.gettempdir()) / "demo.svg"
path.write_text(svg, encoding="utf-8")
print(path, len(svg), "characters")

/tmp/demo.svg 2827 characters


## Diagrams in saved notebooks

Jupyter does not trust the saved outputs of a notebook you did not run yourself, e.g.
one you downloaded or cloned. It removes inline SVG from them, so the diagrams above
show only their words until the cells are run again.

`render_svg_as_img=True` puts each diagram into an `<img>` (the SVG as a base64 data
URI), which Jupyter keeps, so the saved output shows the diagram when the notebook is
opened. Use it for notebooks you share with their outputs.

The flag is **off by default**. GitHub's notebook viewer does the opposite of Jupyter:
it shows inline SVG but blocks image data URIs, so on GitHub the diagram below appears
as a broken image. Also, text in an image cannot be selected or found with Ctrl+F, and
page CSS cannot style it. It only changes the notebook display and
`html_output`; `svg` and `svgs` always return plain SVG.

In [13]:
vis_img = DependencyTreeVisualiser(
    syntax_layer="maltparser_syntax",
    rows=[("morph_analysis", "partofspeech")],
    overlay_layer="ner",
    overlay_label="nertag",
    render_svg_as_img=True,
)
vis_img(text, sentence=0)

In [14]:
html = vis_img.html_output(text, sentence=0)
print(html[:130], "...")

<div style="overflow-x:auto;margin-bottom:12px"><img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC ...


## Other layer names

Layer names are parameters, e.g. for texts from a database with prefixed layers:

```python
DependencyTreeVisualiser(
    syntax_layer="v172_stanza_syntax",
    sentence_layer="v172_sentences",
    overlay_layer="v171_named_entities",
    overlay_label="nertag",
)
```